# Debug — EndoSfMLearner baseline (split oficial endovis)

Notebook minimo para aislar **solo EndoSfMLearner** y reproducir su *baseline* de referencia (AbsRel ~0.070).

Replica el protocolo del `evaluate_depth.py` oficial de AF-SfMLearner (el que el profe usa para todos los modelos):
- Split oficial `splits/endovis/test_files.txt` (550 frames, datasets 1-7)
- GT identico a `export_gt_depth.py`: canal Z del `scene_points{N-1}.tiff`, crop `[0:1024, :]`
- `MIN_DEPTH=1e-3`, `MAX_DEPTH=150`, median scaling
- Inferencia con el `DispResNet` nativo de EndoSLAM (256x832, norm 0.45/0.225, `depth = 1/disp`)

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    ENDOSLAM_PATH = BASE / "EndoSLAM"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE          = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT   = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    ENDOSLAM_PATH = Path("E:/EndoSLAM")
    W             = BASE
    REPO_ROOT     = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE    = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_ENDOSFM       = W / "endosfmlearner_weights" / "11-09-03_58"
ENDOSFM_WEIGHTS = W_ENDOSFM / "dispnet_model_best.pth.tar"
NPZ_CACHE       = BASE / "split_frames.npz"
CAP_MM = 150.0

def load_split(split_file):
    items = []
    with open(split_file) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            folder, frame_id, _side = line.split()
            ds, kf = folder.split("/")
            ds_n = "dataset_" + ds.replace("dataset","")
            kf_n = "keyframe_" + kf.replace("keyframe","")
            items.append((ds_n, kf_n, int(frame_id)))
    return items

SPLIT_ITEMS = load_split(SPLIT_FILE)
print(f"Entorno : {'Colab' if IN_COLAB else 'Local'}")
print(f"Split   : {len(SPLIT_ITEMS)} frames (datasets {sorted({d for d,_,_ in SPLIT_ITEMS})})")
print(f"Pesos   : {'OK' if ENDOSFM_WEIGHTS.exists() else 'NO ENCONTRADO'}  {ENDOSFM_WEIGHTS}")
print(f"NPZ GT  : {'OK' if NPZ_CACHE.exists() else 'NO ENCONTRADO'}  {NPZ_CACHE}")

In [ ]:
import torch, importlib.util

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

_endosfm_dir = ENDOSLAM_PATH / "EndoSfMLearner"
if str(_endosfm_dir) not in sys.path:
    sys.path.insert(0, str(_endosfm_dir))

_spec = importlib.util.spec_from_file_location(
    "_endosfm_models_dbg",
    str(_endosfm_dir / "models" / "__init__.py"),
    submodule_search_locations=[str(_endosfm_dir / "models")])
endosfm_models = importlib.util.module_from_spec(_spec)
sys.modules["_endosfm_models_dbg"] = endosfm_models
_spec.loader.exec_module(endosfm_models)

endosfm = endosfm_models.DispResNet(18, False).to(DEVICE)
w = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm.load_state_dict(w["state_dict"])
endosfm.eval()
print(f"EndoSfMLearner cargado: {sum(p.numel() for p in endosfm.parameters())/1e6:.2f} M params")

In [ ]:
import numpy as np

# Cargar imagenes + GT desde el npz cacheado (mismo que el notebook principal).
# El GT ya viene como canal Z del tiff, crop [0:1024,:], igual que export_gt_depth.py oficial.
def _key(ds, kf, fid): return f"{ds}|{kf}|{fid}"

SPLIT_DATA = {}
assert NPZ_CACHE.exists(), "No existe split_frames.npz — corre primero el notebook principal para generarlo"
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds, kf, fid in SPLIT_ITEMS:
    k = _key(ds, kf, fid)
    ik, gk = "img_"+k, "gt_"+k
    if ik in _npz.files:
        gt = _npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size == 1 and np.isnan(gt).all(): gt = None
        SPLIT_DATA[k] = (_npz[ik], gt)
print(f"Frames cargados: {len(SPLIT_DATA)}")
_n_gt = sum(1 for v in SPLIT_DATA.values() if v[1] is not None)
print(f"Con GT valido  : {_n_gt}")

In [ ]:
from skimage.transform import resize as imresize

def predict_endosfm(img):
    """DispResNet nativo: 256x832, norm (x/255-0.45)/0.225, depth = 1/disp (test_disp.py oficial).

    CRITICO: convertir a float ANTES del imresize. skimage.resize sobre uint8 normaliza a [0,1];
    sobre float preserva el rango 0-255 (como hace imread().astype(float32) en test_disp.py).
    El bug previo (resize sobre uint8 -> /255) dejaba la imagen en ~0-0.004 -> prediccion mala."""
    H, W_ = img.shape[:2]
    r = imresize(img.astype(np.float32), (256, 832)).astype(np.float32)  # float ANTES de resize
    t = torch.from_numpy(((r/255-0.45)/0.225).transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        d = endosfm(t)
    pred_depth = 1.0 / (d.squeeze().detach().cpu().numpy() + 1e-6)
    return imresize(pred_depth, (H, W_))

In [ ]:
# Evaluacion estilo evaluate_depth.py oficial: median scaling, MIN_DEPTH=1e-3, MAX_DEPTH=150
from collections import defaultdict
from tqdm import tqdm

MIN_DEPTH, MAX_DEPTH = 1e-3, 150.0

def compute_errors(gt, pred):
    thresh = np.maximum((gt/pred), (pred/gt))
    a1 = (thresh < 1.25).mean(); a2 = (thresh < 1.25**2).mean(); a3 = (thresh < 1.25**3).mean()
    rmse = np.sqrt(((gt-pred)**2).mean())
    rmse_log = np.sqrt(((np.log(gt)-np.log(pred))**2).mean())
    abs_rel = np.mean(np.abs(gt-pred)/gt)
    sq_rel = np.mean(((gt-pred)**2)/gt)
    return abs_rel, sq_rel, rmse, rmse_log, a1, a2, a3

errors = []; ratios = []; per_ds = defaultdict(list)
for ds, kf, fid in tqdm(SPLIT_ITEMS, desc="EndoSfMLearner"):
    k = _key(ds, kf, fid)
    if k not in SPLIT_DATA: continue
    img, gt = SPLIT_DATA[k]
    if gt is None: continue
    pred = predict_endosfm(img)
    # mask oficial: gt > MIN_DEPTH & gt < MAX_DEPTH (sin crop extra en endovis)
    mask = (gt > MIN_DEPTH) & (gt < MAX_DEPTH) & (~np.isnan(gt))
    if mask.sum() == 0: continue
    pred_m = pred[mask]; gt_m = gt[mask]
    ratio = np.median(gt_m) / np.median(pred_m)
    ratios.append(ratio)
    pred_m = pred_m * ratio
    pred_m[pred_m < MIN_DEPTH] = MIN_DEPTH
    pred_m[pred_m > MAX_DEPTH] = MAX_DEPTH
    e = compute_errors(gt_m, pred_m)
    errors.append(e); per_ds[ds].append(e[0])

mean_errors = np.array(errors).mean(0)
print(f"\nEvaluados: {len(errors)} frames")
print(f"Scaling ratio mediano: {np.median(ratios):.2f}\n")
print(f"{'abs_rel':>9}{'sq_rel':>9}{'rmse':>9}{'rmse_log':>10}{'a1':>8}{'a2':>8}{'a3':>8}")
print(("{:9.4f}"*4 + "{:8.4f}"*3).format(*mean_errors))
print(f"\n>>> AbsRel global EndoSfMLearner: {mean_errors[0]:.4f}  (referencia profe: ~0.0700)\n")
print("AbsRel por dataset:")
for ds in sorted(per_ds):
    print(f"  {ds}: {np.mean(per_ds[ds]):.4f}  (n={len(per_ds[ds])})")